# Qwen3.8-Flash-Next 无审查版（GSQ-RCO abliterated IQ3_S）· llama-server

模型：[SC117/Qwen3.8-Flash-Next-GSQ-RCO-abliterated-GGUF](https://huggingface.co/SC117/Qwen3.8-Flash-Next-GSQ-RCO-abliterated-GGUF) 的 IQ3_S（180B MoE，约 84GB）+ MTP 草稿头。

需要 **G4 运行时**（96GB 显存）。依次运行：
1. 挂载 Drive，从 Hugging Face 下载模型（约 4 分钟）
2. 准备 llama-server（复用 Drive 里的编译结果；不支持这个模型时自动重新编译）
3. 启动服务：默认 1M 上下文（YaRN，可改回 256K 原生）、4 个并发槽位（可调 PARALLEL），端口 8001
4. 保活代理 8000 → 8001（防止长提示词时 Cloudflare 100 秒超时）
5. 建立隧道 → `secrets.json` 里配置的固定域名
6. 对话 + 测速
7. 看日志 / 用完关机
8. **保持运行的监控**：一直运行，防止空闲回收；进程退出时自动重启

后台进程（llama-server、代理、隧道）都用独立进程组启动：点 cell 的停止按钮只停当前 cell，不会连带停掉服务。

实测（G4，1M 上下文 + 8 槽 + KV q8_0，不开 MTP）：显存 84GB；单人约 100 tok/s；8 人同时用总吞吐约 300 tok/s、每人约 40 tok/s，首字 < 1 秒。
旧配置（单槽）4 人同时用时后面的人要排队十几秒。36.8 万 token 输入大海捞针测试通过。


In [ ]:
# 1. 挂载 Drive，从 Hugging Face 下载模型到本机硬盘
from google.colab import drive
drive.mount('/content/drive')

import os, subprocess
HF_REPO = "SC117/Qwen3.8-Flash-Next-GSQ-RCO-abliterated-GGUF"
QUANT = "IQ3_S"                       # 可选 Q2_0 / IQ2_XS / IQ3_XXS / IQ3_S（越往后越大越好）
MODEL_DIR = "/content/models/flash"
MODEL_PATH = f"{MODEL_DIR}/{QUANT}/Qwen3.8-Flash-Next-GSQ-RCO-abliterated-{QUANT}-00001-of-00002.gguf"
MTP_PATH = f"{MODEL_DIR}/mtp-Qwen3.8-Flash-Next-Q8_0.gguf"
MODEL_ALIAS = "qwen3.8-flash-next"
BIN_DIR = "/content/drive/MyDrive/models/bin"   # 编译好的 llama-server 存这里

!nvidia-smi --query-gpu=name,memory.total,compute_cap --format=csv
!pip install -q -U "huggingface_hub[hf_xet]"
os.environ["HF_XET_HIGH_PERFORMANCE"] = "1"      # 多线程高速下载

files = [f"{QUANT}/Qwen3.8-Flash-Next-GSQ-RCO-abliterated-{QUANT}-0000{i}-of-00002.gguf" for i in (1, 2)]
files.append("mtp-Qwen3.8-Flash-Next-Q8_0.gguf")
!hf download {HF_REPO} {" ".join(files)} --local-dir {MODEL_DIR}
!du -sh {MODEL_DIR}/*


In [ ]:
# 2. 准备 llama-server (CUDA)。Drive 里有同架构的编译结果就复用；不支持 Flash-Next（qwen4exp）时重新编译
import os, subprocess
cap = subprocess.check_output(
    ["nvidia-smi", "--query-gpu=compute_cap", "--format=csv,noheader"]).decode().split()[0]
ARCH = cap.replace(".", "")          # H100 -> 90, A100 -> 80
CACHED = f"{BIN_DIR}/llama-server-sm{ARCH}"
SERVER = "/content/llama-server"

# 如果之前因异常中断导致 Drive 里生成了不完整的空文件或损坏的缓存（小于 10MB），我们将其删除重新编译
if os.path.exists(CACHED):
    if not os.path.isfile(CACHED) or os.path.getsize(CACHED) < 10000000:
        print("检测到损坏或不完整的缓存，正在清理并准备重新编译...")
        try:
            if os.path.isdir(CACHED):
                import shutil
                shutil.rmtree(CACHED)
            else:
                os.remove(CACHED)
        except Exception as e:
            print(f"清理缓存失败: {e}")

# 缓存的版本太旧、不支持 Flash-Next 的 qwen4exp 架构时，删掉重编（新版也能跑 27B）
if os.path.isfile(CACHED) and subprocess.run(f'grep -q qwen4exp "{CACHED}"', shell=True).returncode != 0:
    print("Drive 里的 llama-server 不支持 Flash-Next，重新编译")
    os.remove(CACHED)

if os.path.exists(CACHED):
    print("复用 Drive 里的编译结果:", CACHED)
    !cp "{CACHED}" {SERVER} && chmod +x {SERVER}
else:
    print("未检测到可用缓存，开始全新编译...")
    # 清理可能残留的旧目录，防止 git clone 报错
    !rm -rf /content/llama.cpp
    !git clone --depth 1 https://github.com/ggml-org/llama.cpp /content/llama.cpp
    print("正在生成构建配置 (CMake)...")
    !cmake -S /content/llama.cpp -B /content/llama.cpp/build \
        -DGGML_CUDA=ON -DCMAKE_CUDA_ARCHITECTURES={ARCH} \
        -DBUILD_SHARED_LIBS=OFF -DLLAMA_CURL=OFF -DCMAKE_BUILD_TYPE=Release
    print("开始编译 llama-server (这需要几分钟)...")
    # 移除了 tail -3，以便把编译过程的错误或进度完整打印出来，方便定位问题
    !cmake --build /content/llama.cpp/build --target llama-server -j $(nproc)

    if os.path.exists("/content/llama.cpp/build/bin/llama-server"):
        !cp /content/llama.cpp/build/bin/llama-server {SERVER}
        !mkdir -p "{BIN_DIR}" && cp {SERVER} "{CACHED}"
        print("✅ 编译完成并已存到 Drive:", CACHED)
    else:
        raise FileNotFoundError("编译失败：未能成功生成 /content/llama.cpp/build/bin/llama-server，请检查上面的编译日志。")

!{SERVER} --version

In [ ]:
# 3. 后台启动 llama-server（端口 8001，带 API key，只监听本机；对外由第 4 步的保活代理在 8000 端口转发）
import os, time, subprocess, urllib.request, secrets, json

def get_secret(name):
    """先读 Colab Secrets（左侧钥匙图标），再读 Drive 里的 MyDrive/models/secrets.json"""
    try:
        from google.colab import userdata
        v = userdata.get(name)
        if v: return v
    except Exception:
        pass
    try:
        return json.load(open("/content/drive/MyDrive/models/secrets.json")).get(name)
    except Exception:
        return None

if not os.path.exists(MODEL_PATH):
    raise FileNotFoundError(f"找不到 {MODEL_PATH}：请先运行第 1 步下载模型")
subprocess.run(["pkill", "-f", "llama-server"]); time.sleep(2)   # 如果之前启动过，先停掉旧的

API_KEY = get_secret("LLM_API_KEY") or secrets.token_urlsafe(24)   # 配了固定 key 就用固定的
print("API key:", "固定（来自 secrets）" if get_secret("LLM_API_KEY") else "本次随机生成")
# 上下文：原生 256K。官方说明可用 YaRN（factor 4）扩到 1M，但静态 YaRN 会让短文本效果略降，
# 不需要超长文本时可改回 False。1M 时显存约 92GB / 96GB（实测）。
LONG_CONTEXT = True       # True = 1M 上下文（YaRN）；False = 256K 原生
CTX = 1000000 if LONG_CONTEXT else 262144
# 并发：PARALLEL 个请求同时生成（连续批处理）。--kv-unified 让所有槽位共用一个 KV 池，
# 单个请求最长仍可用到 CTX，几个请求一起跑时按实际长度分。
# 实测（G4，每请求 512 token）：8 槽总吞吐约 300 tok/s、每人 40；16 槽总吞吐约 350、每人 23。
PARALLEL = 4
# MTP 投机解码：实测在这个模型上是负收益（单人 96 vs 103 tok/s，8 人 230 vs 300），还多占 7GB 显存，默认关
USE_MTP = False
MTP_ARGS = ["-md", MTP_PATH, "-ngld", "99", "--spec-type", "draft-mtp", "--spec-draft-n-max", "3"] if USE_MTP else []
CTX_ARGS = (["-c", str(CTX), "--rope-scaling", "yarn", "--rope-scale", "4", "--yarn-orig-ctx", "262144"]
            if LONG_CONTEXT else ["-c", str(CTX)])
SERVER_CMD = [SERVER, "-m", MODEL_PATH, "--alias", MODEL_ALIAS, "--api-key", API_KEY,
       "-ngl", "99", *CTX_ARGS, "-np", str(PARALLEL), "--kv-unified",
       "-fa", "on", "-ctk", "q8_0", "-ctv", "q8_0",   # KV 用 8 bit，显存减半，多人长上下文放得下
       # 读长输入时按 1024 token 一块处理，块之间穿插其他人的生成；块太大（4096）时别人会卡到约 1 tok/s
       "-b", "1024", "-ub", "512",
       # 多轮对话的前缀缓存放内存：默认 8GB，长对话一个就 1.5–2.5GB，会被挤掉而整段重算；机器有 176GB
       "--cache-ram", "60000",
       "-lm", "mmap", "--lazy-mode", "on",             # 第 2 个分片（28.8GB n-gram 表）按需读取，不占显存
       *MTP_ARGS,
       "--jinja", "--host", "127.0.0.1", "--port", "8001"]

# 预读 n-gram 表进内存缓存（176GB 内存放得下），避免刚开始并发时卡在读硬盘
subprocess.Popen(f"cat {MODEL_PATH.replace('00001-of', '00002-of')} > /dev/null", shell=True, start_new_session=True)

def start_server():
    p = subprocess.Popen(SERVER_CMD, stdout=open("/content/llama-server.log", "a"), stderr=subprocess.STDOUT, start_new_session=True)
    for i in range(300):
        try:
            if urllib.request.urlopen("http://127.0.0.1:8001/health").status == 200:
                print(f"服务已就绪（{i*2}s）"); return p
        except Exception:
            pass
        if p.poll() is not None:
            print("启动失败，日志："); break
        time.sleep(2)
    return p

open("/content/llama-server.log", "w").close()
proc = start_server()
!grep -E "n_ctx_slot|draft model|listening|error" /content/llama-server.log | tail -n 8
!nvidia-smi --query-gpu=memory.used,memory.total --format=csv


In [ ]:
# 4. 保活代理：8000 → llama-server 8001
#     超长上下文时 llama-server 处理 prompt 可能超过 100 秒，Cloudflare 会因"100 秒无响应"断开（524）。
#     代理在上游 20 秒没响应时先回 200，然后每 15 秒发一次保活（流式发 SSE 注释行，非流式发前导空格），
#     客户端都会自动忽略。隧道、第 4 步、Mac/Magpie 的地址都不用改。
#     注意：已提前回 200 之后，如果 llama-server 报错，状态码仍是 200，错误放在正文里。
import importlib.util
if importlib.util.find_spec("aiohttp") is None:
    !pip install -q aiohttp

PROXY_CODE = r'''"""保活反向代理：挡在 llama-server 前面，避免 Cloudflare 100 秒无响应断开（524）。
上游超过 GRACE 秒没回响应头时，先回 200，然后每 INTERVAL 秒发保活：
  - 流式（SSE）：发 ": keepalive" 注释行，客户端会忽略
  - 非流式（JSON）：在正文前发空格，JSON 解析器会忽略
"""
import asyncio, json, os, sys
from aiohttp import web, ClientSession, ClientTimeout, TCPConnector

LISTEN = int(os.environ.get("PROXY_PORT", "8000"))
UPSTREAM = os.environ.get("UPSTREAM", "http://127.0.0.1:8001")
GRACE = float(os.environ.get("KEEPALIVE_GRACE", "20"))
INTERVAL = float(os.environ.get("KEEPALIVE_INTERVAL", "15"))

HOP = {"host", "content-length", "transfer-encoding", "connection", "keep-alive",
       "accept-encoding", "content-encoding", "upgrade", "te", "trailer", "proxy-connection"}


def log(*a):
    print(*a, file=sys.stderr, flush=True)


async def pump(session, request, body, q):
    """读上游响应，按顺序放进队列：head → data... → end / error"""
    headers = {k: v for k, v in request.headers.items() if k.lower() not in HOP}
    try:
        async with session.request(request.method, UPSTREAM + request.path_qs,
                                   headers=headers, data=body) as resp:
            await q.put(("head", resp.status, resp.headers))
            async for chunk in resp.content.iter_any():
                await q.put(("data", chunk))
        await q.put(("end",))
    except asyncio.CancelledError:
        raise
    except Exception as e:
        await q.put(("error", f"{type(e).__name__}: {e}"))


async def handler(request):
    body = await request.read()
    wants_sse = False
    if body:
        try:
            wants_sse = bool(json.loads(body).get("stream"))
        except Exception:
            pass

    q = asyncio.Queue()
    task = asyncio.create_task(pump(request.app["session"], request, body, q))
    out = None             # 已发给客户端的响应
    sse = wants_sse
    wrote_body = False     # 是否已经转发过上游正文
    at_boundary = True     # SSE：上次转发的内容是否以空行结尾（只在事件之间插保活）

    async def commit(status, ctype, extra=None):
        nonlocal out
        h = {"Content-Type": ctype, "Cache-Control": "no-cache, no-transform",
             "X-Accel-Buffering": "no"}
        h.update(extra or {})
        out = web.StreamResponse(status=status, headers=h)
        await out.prepare(request)

    try:
        while True:
            try:
                item = await asyncio.wait_for(q.get(), GRACE if out is None else INTERVAL)
            except asyncio.TimeoutError:
                if out is None:   # 上游迟迟没回响应头：先回 200，开始保活
                    await commit(200, "text/event-stream" if sse else "application/json")
                    log(f"[keepalive] {request.method} {request.path} 上游 {GRACE:.0f}s 未响应，提前回 200")
                if sse and at_boundary:
                    await out.write(b": keepalive\n\n")
                elif not sse and not wrote_body:
                    await out.write(b" ")
                continue

            kind = item[0]
            if kind == "head":
                _, status, h = item
                ctype = h.get("Content-Type", "application/octet-stream")
                if out is None:
                    sse = "event-stream" in ctype
                    extra = {k: v for k, v in h.items()
                             if k.lower() not in HOP and k.lower() not in
                             ("content-type", "cache-control")}
                    await commit(status, ctype, extra)
                elif status != 200:
                    log(f"[keepalive] 上游返回 {status}，但已提前回了 200，错误内容会放在正文里")
            elif kind == "data":
                chunk = item[1]
                await out.write(chunk)
                wrote_body = True
                at_boundary = chunk.endswith(b"\n\n") or chunk.endswith(b"\r\n\r\n")
            elif kind == "end":
                await out.write_eof()
                return out
            else:  # error
                msg = item[1]
                log(f"[keepalive] 上游出错: {msg}")
                err = {"error": {"message": f"proxy upstream error: {msg}", "type": "proxy_error"}}
                if out is None:
                    return web.json_response(err, status=502)
                payload = json.dumps(err).encode()
                await out.write((b"data: " + payload + b"\n\n") if sse else
                                (payload if not wrote_body else b""))
                await out.write_eof()
                return out
    except ConnectionResetError:
        log(f"[keepalive] 客户端断开 {request.method} {request.path}")
        return out
    except asyncio.CancelledError:
        log(f"[keepalive] 客户端断开 {request.method} {request.path}")
        raise
    finally:
        task.cancel()   # 客户端断开时也断开上游，llama-server 会停止这次生成


async def on_startup(app):
    app["session"] = ClientSession(timeout=ClientTimeout(total=None, sock_read=None),
                                   connector=TCPConnector(limit=0), auto_decompress=False)


async def on_cleanup(app):
    await app["session"].close()


app = web.Application(client_max_size=1024 ** 3)   # 长上下文请求体可能很大
app.on_startup.append(on_startup)
app.on_cleanup.append(on_cleanup)
app.router.add_route("*", "/{tail:.*}", handler)

if __name__ == "__main__":
    log(f"keepalive proxy :{LISTEN} -> {UPSTREAM} (grace {GRACE}s, interval {INTERVAL}s)")
    web.run_app(app, host="127.0.0.1", port=LISTEN, access_log=None, print=None)
'''
open("/content/keepalive_proxy.py", "w").write(PROXY_CODE)

subprocess.run(["pkill", "-f", "keepalive_proxy.py"]); time.sleep(1)   # 停掉旧代理
plog = open("/content/keepalive_proxy.log", "w")
proxy_proc = subprocess.Popen(["python3", "/content/keepalive_proxy.py"],
                              env={**os.environ, "PROXY_PORT": "8000", "UPSTREAM": "http://127.0.0.1:8001"},
                              stdout=plog, stderr=subprocess.STDOUT, start_new_session=True)
ok = False
for _ in range(30):
    try:
        ok = urllib.request.urlopen("http://127.0.0.1:8000/health", timeout=5).status == 200
        if ok: break
    except Exception:
        pass
    if proxy_proc.poll() is not None: break
    time.sleep(1)
if ok:
    print("✅ 保活代理已就绪：127.0.0.1:8000 → llama-server :8001")
else:
    print("❌ 代理没起来（或 llama-server 没在 8001 运行），日志：")
    !tail -n 20 /content/keepalive_proxy.log


In [ ]:
# 5. 建立 Cloudflare 隧道 （读取 Drive 里的 models/secrets.json，用 scripts/setup_tunnel.sh 生成）
#    没配 secrets 时自动退回临时 trycloudflare 地址
import re
subprocess.run(["pkill", "-f", "cloudflared"])   # 停掉旧隧道（用 Python 调，避免 pkill 误杀自己所在的 shell）
if not os.path.exists("/content/cloudflared"):
    urllib.request.urlretrieve(
        "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64",
        "/content/cloudflared")
    os.chmod("/content/cloudflared", 0o755)
tlog = open("/content/cloudflared.log", "w")
CF_TOKEN, CF_HOST = get_secret("CF_TUNNEL_TOKEN"), get_secret("CF_HOSTNAME")

if CF_TOKEN and CF_HOST:
    TUNNEL_CMD = ["/content/cloudflared", "tunnel", "--no-autoupdate", "--protocol", "http2",
                  "--url", "http://127.0.0.1:8000", "run", "--token", CF_TOKEN]
    tunnel = subprocess.Popen(TUNNEL_CMD, stdout=tlog, stderr=subprocess.STDOUT, start_new_session=True)
    PUBLIC_URL = f"https://{CF_HOST}"
    for _ in range(30):
        if "Registered tunnel connection" in open("/content/cloudflared.log").read(): break
        time.sleep(1)
    else:
        print("固定隧道可能没连上，日志："); PUBLIC_URL = None
        !tail -n 20 /content/cloudflared.log
else:
    print("⚠️ 没读到 secrets.json，使用临时地址")
    TUNNEL_CMD = ["/content/cloudflared", "tunnel", "--no-autoupdate", "--protocol", "http2",
                  "--url", "http://127.0.0.1:8000"]
    tunnel = subprocess.Popen(TUNNEL_CMD, stdout=tlog, stderr=subprocess.STDOUT, start_new_session=True)
    PUBLIC_URL = None
    for _ in range(60):
        m = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", open("/content/cloudflared.log").read())
        if m: PUBLIC_URL = m.group(0); break
        time.sleep(1)
    if not PUBLIC_URL:
        print("隧道没起来，日志：")
        !tail -n 20 /content/cloudflared.log

if PUBLIC_URL:
    fixed = bool(CF_TOKEN and CF_HOST and get_secret("LLM_API_KEY"))
    print("✅ 固定地址 + 固定 key，本地 .env 不用改\n" if fixed
          else "把下面三行覆盖写入 Mac 上的 ~/探索/colab/.env ：\n")
    print(f"COLAB_LLM_URL={PUBLIC_URL}/v1")
    if not fixed: print(f"COLAB_LLM_KEY={API_KEY}")
    print(f"COLAB_LLM_MODEL={MODEL_ALIAS}")
    # 自检：从公网地址访问一次
    time.sleep(5)
    try:
        r = urllib.request.urlopen(urllib.request.Request(PUBLIC_URL + "/v1/models",
                headers={"Authorization": "Bearer " + API_KEY,
                         "User-Agent": "colab-llm/1.0"}), timeout=20)
        print("公网自检:", r.status, json.loads(r.read())["data"][0]["id"])
    except Exception as e:
        print("公网自检失败（可再等几十秒重试第 5 步）:", e)

In [ ]:
# 6. 对话 + 测速（流式）
!pip install -q openai
import time
from openai import OpenAI
client = OpenAI(base_url="http://127.0.0.1:8000/v1", api_key=API_KEY)

def chat(prompt, max_tokens=1024):
    t0, first, n = time.time(), None, 0
    stream = client.chat.completions.create(
        model=MODEL_ALIAS, stream=True, max_tokens=max_tokens,
        messages=[{"role": "user", "content": prompt}],
        stream_options={"include_usage": True})
    for chunk in stream:
        if chunk.usage: n = chunk.usage.completion_tokens
        if not chunk.choices: continue
        d = chunk.choices[0].delta
        text = getattr(d, "reasoning_content", None) or d.content or ""
        if text and first is None: first = time.time()
        print(text, end="", flush=True)
    dt = time.time() - (first or t0)
    print(f"\n\n--- 首字 {(first or t0)-t0:.2f}s | {n} tokens | {n/dt:.1f} tok/s ---")

chat("用三句话介绍一下你自己。")

In [ ]:
# 7.（可选）看日志
!tail -n 30 /content/llama-server.log
!tail -n 10 /content/keepalive_proxy.log
!tail -n 10 /content/cloudflared.log

# 用完关机：先停掉第 8 步的监控，再取消下面两行注释并运行，立刻释放 GPU，停止消耗计算单元
# from google.colab import runtime
# runtime.unassign()


In [ ]:
# 8. 保持运行的监控：最后运行，让它一直跑着（cell 一直在执行，Colab 就不会因为「空闲」回收运行时；
#    API 请求本身不算活动）。每 5 分钟自检一次；进程退出或连续 3 次无响应时自动重启。
#    想停时点 cell 左边的停止按钮（只停监控，服务和隧道继续运行）。
import time, urllib.request, subprocess, os

def health(url, key=None):
    try:
        h = {"User-Agent": "colab-llm-monitor/1.0"}   # Cloudflare 会拦 Python 默认 UA（403 / error 1010）
        if key: h["Authorization"] = f"Bearer {key}"
        return urllib.request.urlopen(urllib.request.Request(url, headers=h), timeout=30).status == 200
    except Exception:
        return False

bad = 0
while True:
    msgs = []
    ok_srv = proc.poll() is None and health("http://127.0.0.1:8001/health")
    bad = 0 if ok_srv else bad + 1
    if proc.poll() is not None or bad >= 3:          # 处理超长输入时可能短暂无响应，连续 3 次才重启
        msgs.append("llama-server 异常，重启"); subprocess.run(["pkill", "-f", "llama-server"]); time.sleep(3)
        proc = start_server(); bad = 0
    elif bad:
        msgs.append(f"llama-server 无响应（第 {bad} 次，可能在处理长输入）")
    if proxy_proc.poll() is not None:
        msgs.append("保活代理退出，重启")
        proxy_proc = subprocess.Popen(["python3", "/content/keepalive_proxy.py"],
                                      env={**os.environ, "PROXY_PORT": "8000", "UPSTREAM": "http://127.0.0.1:8001"},
                                      stdout=open("/content/keepalive_proxy.log", "a"), stderr=subprocess.STDOUT,
                                      start_new_session=True)
    if tunnel.poll() is not None:
        msgs.append("隧道退出，重启")
        tunnel = subprocess.Popen(TUNNEL_CMD, stdout=open("/content/cloudflared.log", "a"), stderr=subprocess.STDOUT,
                                  start_new_session=True)
    public = health(PUBLIC_URL + "/v1/models", API_KEY) if PUBLIC_URL else None
    used = subprocess.run(["nvidia-smi", "--query-gpu=memory.used", "--format=csv,noheader"],
                          capture_output=True, text=True).stdout.strip()
    status = "；".join(msgs) if msgs else "llama-server + 代理 + 隧道正常"
    print(f"{time.strftime('%m-%d %H:%M')}  {status} | 公网 {'OK' if public else ('失败' if public is False else '-')} | 显存 {used}", flush=True)
    time.sleep(300)
